# Stage 6-1: CLI 스크립트와 배치 실험

이 노트북은 `scripts/` CLI 스크립트를 노트북 셀에서 직접 호출하는 방법과
`experiments/run_all.py`를 통한 배치 실험 실행 흐름을 실습한다.

**학습 구조**
```
scripts/train.py   : 단일 실험 학습 (CLI 직접 실행)
scripts/evaluate.py: 저장된 checkpoint 평가
experiments/run_all.py: 전체 6개 설정 배치 실행
    -> outputs/{exp_name}/model.npz  (checkpoint)
```

In [ ]:
# sys.path 설정 — 프로젝트 루트를 모듈 검색 경로에 추가한다.
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

In [ ]:
import subprocess
import matplotlib.pyplot as plt

## 1. scripts/train.py 직접 호출

`scripts/train.py`는 `argparse`로 task, model, epochs, lr, batch_size를 받아
학습 루프를 실행하고 checkpoint를 저장한다.

```
--task 인자 → get_task_spec(task) → task_spec dict
    → Trainer(model, optimizer, task_spec)
    → Evaluator(model, task_spec)
```

노트북에서는 `subprocess.run`으로 CLI를 직접 호출하여 stdout을 캡처한다.

In [ ]:
# scripts/train.py — multiclass MLP 3 epoch (소규모 실행)
DATASET_DIR = "/mnt/d/datasets/mnist"
CHECKPOINT = "outputs/multiclass_mlp_ep3_lr0.01_bs64/model.npz"

result = subprocess.run(
    [
        sys.executable, "../../scripts/train.py",
        "--task", "multiclass",
        "--model", "mlp",
        "--epochs", "3",
        "--lr", "0.01",
        "--batch_size", "64",
        "--dataset_dir", DATASET_DIR,
        "--checkpoint", CHECKPOINT,
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr[:200])

In [ ]:
# checkpoint 파일 생성 확인
if os.path.exists(CHECKPOINT):
    size_kb = os.path.getsize(CHECKPOINT) / 1024
    print(f"checkpoint 생성 확인: {CHECKPOINT} ({size_kb:.1f} KB)")
else:
    print("checkpoint 없음 — train.py 실행 결과 확인 필요")

## 2. scripts/evaluate.py — checkpoint 로드 평가

`scripts/evaluate.py`는 저장된 checkpoint를 불러와 test set에 대한 loss와 metric을 출력한다.
`--checkpoint`를 지정하지 않으면 초기화된 파라미터로 평가한다.

In [ ]:
result_ev = subprocess.run(
    [
        sys.executable, "../../scripts/evaluate.py",
        "--task", "multiclass",
        "--model", "mlp",
        "--batch_size", "256",
        "--dataset_dir", DATASET_DIR,
        "--checkpoint", CHECKPOINT,
    ],
    capture_output=True,
    text=True,
)
print(result_ev.stdout)
if result_ev.stderr:
    print("[stderr]", result_ev.stderr[:200])

## 3. exp_name 규칙

`experiments/` 스크립트는 `exp_name`으로 실험 설정을 파일명에 인코딩한다.

```
{task}_{model}_ep{epochs}_lr{lr}_bs{batch_size}
예: multiclass_mlp_ep10_lr0.01_bs64
```

`outputs/{exp_name}/model.npz`에 checkpoint가 저장된다.

In [ ]:
# exp_name 생성 규칙 확인
def exp_name(cfg):
    return f"{cfg['task']}_{cfg['model']}_ep{cfg['epochs']}_lr{cfg['lr']}_bs{cfg['batch_size']}"

CONFIGS = [
    {"task": "multiclass", "model": "mlp", "epochs": 10, "lr": 0.01, "batch_size": 64},
    {"task": "multiclass", "model": "cnn", "epochs": 10, "lr": 0.001, "batch_size": 32},
    {"task": "binary",     "model": "mlp", "epochs": 10, "lr": 0.01, "batch_size": 64},
    {"task": "binary",     "model": "cnn", "epochs": 10, "lr": 0.001, "batch_size": 32},
    {"task": "regression", "model": "mlp", "epochs": 10, "lr": 0.01, "batch_size": 64},
    {"task": "regression", "model": "cnn", "epochs": 10, "lr": 0.001, "batch_size": 32},
]

print("실험 설정 목록:")
for cfg in CONFIGS:
    name = exp_name(cfg)
    checkpoint = f"outputs/{name}/model.npz"
    exists = "[O]" if os.path.exists(f"../../{checkpoint}") else "[ ]"
    print(f"  {exists} {name}")

## 4. experiments/run_all.py — 배치 실험 실행

`experiments/run_all.py`는 6개 설정에 대해 train → evaluate → predict → visualize를
순서대로 실행하는 배치 스크립트이다.

**실행 시간 주의**: MLP 6개 설정 × 10 epoch 학습에 수 분이 소요된다.
CNN이 포함된 경우 CuPy 환경(`cupy_py311_cuda118`)에서 실행해야 한다.

아래 셀은 이미 `outputs/` 결과가 없을 경우 실행한다.

In [ ]:
# run_all.py 실행 여부 확인 — 결과 파일이 없을 때만 실행한다.
sample_output = "../../outputs/multiclass_mlp_ep10_lr0.01_bs64/model.npz"

if not os.path.exists(sample_output):
    print("학습 결과 없음 — run_all.py 실행 중... (수 분 소요)")
    result_all = subprocess.run(
        [sys.executable, "../../experiments/run_all.py"],
        capture_output=True,
        text=True,
    )
    print(result_all.stdout[-2000:])
    if result_all.stderr:
        print("[stderr]", result_all.stderr[:500])
else:
    print("학습 결과 존재 — run_all.py 실행 건너뜀")
    print(f"  {sample_output}")

In [ ]:
# 전체 실험 결과 checkpoint 존재 확인
print("outputs/ 폴더 구조:")
outputs_dir = "../../outputs"
if os.path.exists(outputs_dir):
    for name in sorted(os.listdir(outputs_dir)):
        npz_path = os.path.join(outputs_dir, name, "model.npz")
        exists = "[O]" if os.path.exists(npz_path) else "[ ]"
        print(f"  {exists} outputs/{name}/")
else:
    print("  outputs/ 폴더 없음 — run_all.py 실행 필요")

## 5. src/core/ 조립 흐름 복습

`scripts/train.py`는 `src/core/` 실행 객체를 직접 조립한다.
아래는 스크립트와 동일한 흐름을 노트북에서 재현한 것이다.

In [ ]:
from src.data.mnist import MnistDataset, get_task_spec
from src.data.dataloader import DataLoader
from src.models.mlp import MLP
from src.core.optimizers import SGD
from src.core.trainer import Trainer
from src.core.evaluator import Evaluator
from src.utils import checkpoints

task = "multiclass"
ts = get_task_spec(task)
train_ds = MnistDataset("train", task, dataset_dir=DATASET_DIR)
test_ds = MnistDataset("test", task, dataset_dir=DATASET_DIR)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

model = MLP(task=task, seed=42)
optimizer = SGD(model, lr=0.01)
trainer = Trainer(model, optimizer, ts)
evaluator = Evaluator(model, ts)

logs = []
for epoch in range(1, 4):
    tr = trainer.fit(train_loader)
    te = evaluator.evaluate(test_loader)
    logs.append({"epoch": epoch, "train": tr, "test": te})
    print(f"Epoch {epoch} | train loss={tr['loss']:.4f} metric={tr['metric']:.4f}"
          f" | test loss={te['loss']:.4f} metric={te['metric']:.4f}")

In [ ]:
# 학습 곡선 시각화
from src.utils.training_plots import plot_training_log
import tempfile

tmpdir = tempfile.mkdtemp()
plot_path = plot_training_log(logs, output_dir=tmpdir, filename="demo.png")

from PIL import Image
plt.figure(figsize=(12, 4))
plt.imshow(Image.open(plot_path))
plt.axis("off")
plt.title("scripts/train.py 동일 흐름 (3 epoch)")
plt.tight_layout()
plt.show()

## 6. 요약

| 스크립트 | 역할 | 주요 인자 |
|---|---|---|
| `scripts/train.py` | 단일 실험 학습 + checkpoint 저장 | `--task`, `--model`, `--epochs`, `--lr`, `--batch_size` |
| `scripts/evaluate.py` | checkpoint 로드 후 test set 평가 | `--checkpoint` |
| `experiments/run_all.py` | 6개 설정 배치 실행 | 설정 내장 (CONFIGS) |

**exp_name 규칙**
```
{task}_{model}_ep{epochs}_lr{lr}_bs{batch_size}
예: multiclass_mlp_ep10_lr0.01_bs64
```

**스크립트 조립 원칙**
- `scripts/`는 `src/core/` 실행 객체만 참조한다
- `task_spec`은 `get_task_spec(task)`가 반환하고 `Trainer`, `Evaluator`에 전달된다
- checkpoint는 `checkpoints.save/load`로 `.npz` 형식으로 관리한다